# PRR retrieval inference on the frozen aligned protocol (24 dataset-horizon cells)

This notebook reconstructs the main **Predictive Relevance Retrieval (PRR)** output on the 24 long-horizon retrieval cells.

## Inputs

- frozen common query manifests;
- 24 trained PRR full-stack checkpoints (one per dataset x horizon).

The companion `01_full_prr_training.ipynb` reproduces the checkpoint training recipe.
This inference notebook first looks for reproduced stacks under
`artifacts/full_prr_training/full_stacks/` and otherwise falls back to the original experiment paths.

## PRR inference

For each query, PRR uses:

1. Pearson Top-100 candidate retrieval;
2. learned-representation Top-100 candidate retrieval;
3. the union of both supports;
4. the trained wide fusion reranker using statistical and learned representation features;
5. Top-10 historical futures.

No forecasting backbone is trained or executed here. The primary output is the PRR retrieval cache and its AnalogFutureMSE. Downstream trust calibration is handled separately.

In [ ]:

# Cell 1 — configuration and paths

from pathlib import Path
import os, sys, re, json, time, gc, math, random, hashlib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path(os.environ.get("PRR_ROOT", str(Path.cwd().resolve().parents[1] if Path.cwd().name == "prr_long_horizon" else Path.cwd().resolve())))
EXP33 = ROOT / "results" / "exp33"
EXP34 = ROOT / "results" / "exp34"

MANIFEST_ROOT = EXP33 / "final_eval_manifest"
DIRECT_ROOT = EXP34 / "patchtst_direct_cache"

RESULT_ROOT = ROOT / "results" / "exp35B"
PART_ROOT = RESULT_ROOT / "channel_parts"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
PART_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)

CHANNELS = {
    "ETTh1": 7,
    "Weather": 21,
    "Electricity": 321,
    "Traffic": 862,
    "Exchange": 8,
    "Solar": 137,
}

DEVICE = torch.device("cuda:0")
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))
print("Result:", RESULT_ROOT)

class CFG:
    retrieval_seq_len = 96
    memory_stride = 8
    top_k = 10
    top_m_pattern = 100
    top_m_learned = 100
    union_max = 200
    emb_dim = 64
    enc_hidden = 128
    rerank_hidden = 384
    rerank_dropout = 0.1
    feature_clip = 8.0
    seed = 0
    query_batch = 64

cfg = CFG()
EPS = 1e-8


In [ ]:
# Cell 2 — resolve the 24 PRR full-stack paths

REVIEWER_STACK_ROOT = ROOT / "artifacts" / "full_prr_training" / "full_stacks"

def legacy_stack_path(dataset, horizon):
    if dataset == "ETTh1":
        return (
            ROOT
            / "checkpoints"
            / "11_etth1_all_horizons_five_backbones_fixed_alpha01"
            / "full_stacks"
            / f"ETTh1_H{int(horizon)}_Full.pt"
        )
    return (
        ROOT
        / "checkpoints"
        / "12_multidataset_screening_itransformer_segmoe_fixed_alpha01"
        / "full_stacks"
        / f"{dataset}_H{int(horizon)}_screen.pt"
    )

def reviewer_stack_path(dataset, horizon):
    suffix = "Full" if dataset == "ETTh1" else "screen"
    return REVIEWER_STACK_ROOT / f"{dataset}_H{int(horizon)}_{suffix}.pt"

STACKS = {}
rows = []

for d in DATASETS:
    for h in HORIZONS:
        reviewer = reviewer_stack_path(d, h)
        legacy = legacy_stack_path(d, h)

        if reviewer.is_file():
            chosen = reviewer
            source = "reviewer_training"
        elif legacy.is_file():
            chosen = legacy
            source = "legacy_experiment"
        else:
            chosen = reviewer
            source = "missing"

        STACKS[(d, h)] = chosen
        rows.append({
            "dataset": d,
            "horizon": h,
            "exists": chosen.exists(),
            "source": source,
            "size_MB": chosen.stat().st_size / 2**20 if chosen.exists() else np.nan,
            "path": str(chosen),
        })

stack_inventory = pd.DataFrame(rows)
display(stack_inventory)

assert len(stack_inventory) == 24
if not stack_inventory.exists.all():
    missing = stack_inventory[~stack_inventory.exists]
    raise FileNotFoundError(
        "Missing PRR full-stack checkpoints. Run 01_full_prr_training.ipynb "
        "or set PRR_ROOT to a repository containing the original checkpoints.\n"
        + missing.to_string(index=False)
    )

print("[PASS] PRR full-stack coverage = 24/24")

In [ ]:

# Cell 3 — raw dataset paths and train-only normalization

DATA_CANDIDATES = {
    "ETTh1": [
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
    ],
    "Weather": [
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
    ],
    "Electricity": [
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
    ],
    "Traffic": [
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
    ],
    "Exchange": [
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
    ],
    "Solar": [
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
    ],
}

def resolve_data(dataset):
    p=next((p for p in DATA_CANDIDATES[dataset] if p.exists()),None)
    if p is None:
        raise FileNotFoundError(dataset)
    return p

def read_raw(dataset,path):
    if dataset=="Solar":
        try:
            df=pd.read_csv(path,header=None)
            if df.shape[1]==1:
                df=pd.read_csv(path,header=None,sep=r"\s+")
        except Exception:
            df=pd.read_csv(path,header=None,sep=r"\s+")
    else:
        df=pd.read_csv(path)
        if "date" in df.columns:
            df=df.drop(columns=["date"])
        if "OT" in df.columns:
            df=df[[x for x in df.columns if x!="OT"]+["OT"]]

    df=(
        df.apply(pd.to_numeric,errors="coerce")
        .dropna(axis=1,how="all")
        .replace([np.inf,-np.inf],np.nan)
        .interpolate(axis=0,limit_direction="both")
        .ffill()
        .bfill()
    )
    return df.to_numpy(dtype=np.float32)

def bounds(dataset,n):
    if dataset=="ETTh1":
        train_end=12*30*24
        val_end=train_end+4*30*24
        test_end=n
    else:
        train_end=int(.70*n)
        val_end=n-int(.20*n)
        test_end=n
    return train_end,val_end,test_end

DATA={}

for d in DATASETS:
    p=resolve_data(d)
    raw=read_raw(d,p)
    assert raw.shape[1]==CHANNELS[d], (d,raw.shape)

    train_end,val_end,test_end=bounds(d,len(raw))
    mu=raw[:train_end].mean(axis=0).astype(np.float32)
    sd=raw[:train_end].std(axis=0,ddof=0).astype(np.float32)
    assert np.all(sd>1e-6)

    z=((raw-mu[None,:])/sd[None,:]).astype(np.float32)

    DATA[d]={
        "path":p,
        "raw":raw,
        "z":z,
        "train_end":train_end,
        "val_end":val_end,
        "test_end":test_end,
    }

    print(
        f"{d:11s}: rows={len(raw):6d} C={raw.shape[1]:4d} "
        f"train={train_end} val={val_end}"
    )

print("[PASS] all six datasets normalized from training only")


In [ ]:

# Cell 4 — exact PRR architecture and helper functions

def seed_everything(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def batch_pattern(x):
    x=np.asarray(x,np.float32)
    xc=x-x.mean(axis=-1,keepdims=True)
    n=np.linalg.norm(xc,axis=-1,keepdims=True)
    return np.where(
        n>EPS,
        xc/np.maximum(n,EPS),
        0.0
    ).astype(np.float32)

def context7(x):
    x=np.asarray(x,np.float32)
    short=max(8,cfg.retrieval_seq_len//4)
    m=x.mean(axis=-1)
    s=x.std(axis=-1)+EPS

    f1=(x[...,-1]-m)/s
    f2=(x[...,-short:].mean(axis=-1)-m)/s
    f3=(x[...,-1]-x[...,-short])/s
    f4=(x[...,-1]-x[...,0])/s

    df=np.diff(x,axis=-1)
    ds=np.diff(x[...,-short:],axis=-1)
    f5=(ds.std(axis=-1)+EPS)/(df.std(axis=-1)+EPS)

    t=np.linspace(-1.,1.,cfg.retrieval_seq_len,dtype=np.float32)
    t=t-t.mean()
    f6=(
        np.sum(t*(x-m[...,None]),axis=-1)
        /(np.sum(t*t)+EPS)
    )/s

    a=x[...,:-1]
    b=x[...,1:]
    a=a-a.mean(axis=-1,keepdims=True)
    b=b-b.mean(axis=-1,keepdims=True)
    f7=(
        np.sum(a*b,axis=-1)
        /(
            np.sqrt(np.sum(a*a,axis=-1)*np.sum(b*b,axis=-1))
            +EPS
        )
    )

    return np.stack([f1,f2,f3,f4,f5,f6,f7],axis=-1).astype(np.float32)

class TemporalEncoder1D(nn.Module):
    def __init__(self,hidden=128,emb_dim=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv1d(1,hidden//2,5,stride=2,padding=2),
            nn.GELU(),
            nn.Conv1d(hidden//2,hidden,5,stride=2,padding=2),
            nn.GELU(),
            nn.Conv1d(hidden,hidden,3,stride=2,padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj=nn.Linear(hidden,emb_dim)

    def forward(self,x):
        if x.ndim==2:
            x=x[:,None,:]
        h=self.net(x).squeeze(-1)
        return F.normalize(self.proj(h),dim=-1)

def inverse_softplus(x):
    return math.log(math.exp(float(x))-1.)

class WideFusionReranker(nn.Module):
    def __init__(self):
        super().__init__()
        in_dim=1+4*(7+cfg.emb_dim)
        h=cfg.rerank_hidden
        self.net=nn.Sequential(
            nn.Linear(in_dim,h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h,h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h,1),
        )
        self.raw_alpha=nn.Parameter(
            torch.tensor(inverse_softplus(.1),dtype=torch.float32)
        )

    def forward(self,feat,base,mask):
        delta=self.net(feat).squeeze(-1)
        alpha=F.softplus(self.raw_alpha)
        score=base+alpha*delta
        return score.masked_fill(~mask,-1e9)

@torch.no_grad()
def encode_rows(enc,x,batch=512):
    out=[]
    enc.eval()
    for left in range(0,len(x),batch):
        t=torch.from_numpy(x[left:left+batch]).float().to(DEVICE)
        out.append(enc(t).cpu().numpy())
    return np.concatenate(out,axis=0).astype(np.float32)

class UpgradedStack:
    def __init__(self,qenc,henc,reranker,stats_med,stats_iqr,
                 dataset,horizon,training_channels,
                 training_query_anchors,rerank_best_epoch):
        self.qenc=qenc.eval()
        self.henc=henc.eval()
        self.reranker=reranker.eval()
        self.stats_med=np.asarray(stats_med,np.float32)
        self.stats_iqr=np.asarray(stats_iqr,np.float32)
        self.dataset=str(dataset)
        self.horizon=int(horizon)
        self.training_channels=np.asarray(training_channels,np.int64)
        self.training_query_anchors=np.asarray(training_query_anchors,np.int64)
        self.rerank_best_epoch=int(rerank_best_epoch)

    def scale_stats(self,x):
        return np.clip(
            (x-self.stats_med)/self.stats_iqr,
            -cfg.feature_clip,
            cfg.feature_clip
        ).astype(np.float32)

def safe_torch_load(path):
    try:
        return torch.load(path,map_location=DEVICE,weights_only=False)
    except TypeError:
        return torch.load(path,map_location=DEVICE)

def infer_stack_identity_from_path(path):
    path = Path(path)
    s = str(path)

    dataset = None
    for d in DATASETS:
        if d.lower() in s.lower():
            dataset = d
            break

    m = re.search(r"_H(96|192|336|720)(?:_|\\.|$)", path.name, re.I)
    horizon = int(m.group(1)) if m else None

    return dataset, horizon

def load_stack(path, expected_dataset=None, expected_horizon=None):
    obj = safe_torch_load(path)

    # Only these keys are required for inference.
    required_inference = {
        "query_encoder",
        "history_encoder",
        "reranker",
        "stats_med",
        "stats_iqr",
        "rerank_best_epoch",
    }

    missing = required_inference - set(obj)
    if missing:
        raise KeyError(
            f"{path}: missing inference-critical keys {sorted(missing)}"
        )

    # Older Experiment-11 ETTh1 stacks predate the richer metadata schema.
    # Those missing fields are NOT required by retrieval inference.
    path_dataset, path_horizon = infer_stack_identity_from_path(path)

    dataset = obj.get(
        "dataset",
        expected_dataset if expected_dataset is not None else path_dataset
    )

    horizon = obj.get(
        "horizon",
        expected_horizon if expected_horizon is not None else path_horizon
    )

    if dataset is None:
        raise KeyError(f"{path}: could not infer dataset identity")

    if horizon is None:
        raise KeyError(f"{path}: could not infer horizon identity")

    training_channels = np.asarray(
        obj.get("training_channels", []),
        dtype=np.int64,
    )

    training_query_anchors = np.asarray(
        obj.get("training_query_anchors", []),
        dtype=np.int64,
    )

    seed_everything(cfg.seed)

    qenc = TemporalEncoder1D(
        cfg.enc_hidden,
        cfg.emb_dim,
    ).to(DEVICE)

    henc = TemporalEncoder1D(
        cfg.enc_hidden,
        cfg.emb_dim,
    ).to(DEVICE)

    reranker = WideFusionReranker().to(DEVICE)

    qenc.load_state_dict(
        obj["query_encoder"],
        strict=True,
    )

    henc.load_state_dict(
        obj["history_encoder"],
        strict=True,
    )

    reranker.load_state_dict(
        obj["reranker"],
        strict=True,
    )

    return UpgradedStack(
        qenc,
        henc,
        reranker,
        obj["stats_med"],
        obj["stats_iqr"],
        dataset,
        int(horizon),
        training_channels,
        training_query_anchors,
        obj["rerank_best_epoch"],
    )

def topk_np(scores,k):
    k=min(int(k),len(scores))
    part=np.argpartition(-scores,k-1)[:k]
    return part[np.argsort(-scores[part])]

print("[PASS] PRR implementation loaded")


In [ ]:
# Schema diagnostic — older ETTh1 stacks may omit non-inference metadata

schema_rows = []

for d in DATASETS:
    for h in HORIZONS:
        p = STACKS[(d,h)]
        obj = safe_torch_load(p)

        schema_rows.append({
            "dataset": d,
            "horizon": h,
            "n_keys": len(obj) if isinstance(obj, dict) else np.nan,
            "has_dataset": isinstance(obj, dict) and "dataset" in obj,
            "has_horizon": isinstance(obj, dict) and "horizon" in obj,
            "has_training_channels": isinstance(obj, dict) and "training_channels" in obj,
            "has_training_query_anchors": isinstance(obj, dict) and "training_query_anchors" in obj,
            "has_query_encoder": isinstance(obj, dict) and "query_encoder" in obj,
            "has_history_encoder": isinstance(obj, dict) and "history_encoder" in obj,
            "has_reranker": isinstance(obj, dict) and "reranker" in obj,
            "path": str(p),
        })

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

critical = (
    schema_df.has_query_encoder
    & schema_df.has_history_encoder
    & schema_df.has_reranker
)

assert critical.all()

print("[PASS] 24/24 stacks contain inference-critical model weights.")
print("[INFO] Missing legacy metadata is allowed and reconstructed from the experiment condition/path.")


In [ ]:

# Cell 5 — verify all 24 stacks strict-load and identity

audit=[]

for d in DATASETS:
    for h in HORIZONS:
        t0=time.perf_counter()
        stack=load_stack(STACKS[(d,h)], expected_dataset=d, expected_horizon=h)

        ok_dataset=(stack.dataset.lower()==d.lower())
        ok_h=(stack.horizon==h)

        audit.append({
            "dataset":d,
            "horizon":h,
            "stack_dataset":stack.dataset,
            "stack_horizon":stack.horizon,
            "training_channels":len(stack.training_channels),
            "training_queries":len(stack.training_query_anchors),
            "rerank_best_epoch":stack.rerank_best_epoch,
            "identity_ok":ok_dataset and ok_h,
            "load_seconds":time.perf_counter()-t0,
        })

        del stack
        gc.collect()
        torch.cuda.empty_cache()

audit_df=pd.DataFrame(audit)
display(audit_df)

assert audit_df.identity_ok.all()
print("[PASS] 24/24 PRR stacks strict-loaded with correct identities")


In [ ]:

# Cell 6 — direct cache + manifest loaders

def load_manifest(d,h,split):
    p=MANIFEST_ROOT/f"{d}_H{h}_{split}.npz"
    assert p.exists(),p
    with np.load(p,allow_pickle=False) as z:
        return (
            np.asarray(z["anchor"],dtype=np.int64),
            np.asarray(z["channel"],dtype=np.int64),
        )

def load_direct(d,h,split):
    p=DIRECT_ROOT/f"{d}_H{h}_{split}_patchtst_direct.npz"
    assert p.exists(),p

    with np.load(p,allow_pickle=False) as z:
        a=np.asarray(z["anchor"],np.int64)
        c=np.asarray(z["channel"],np.int64)
        direct=np.asarray(z["direct"],np.float32)
        truth=np.asarray(z["truth"],np.float32)

    ma,mc=load_manifest(d,h,split)

    assert np.array_equal(a,ma)
    assert np.array_equal(c,mc)
    assert direct.shape==truth.shape==(len(a),h)

    return a,c,direct,truth

for dataset in DATASETS:
    for h in HORIZONS:
        for split in ("val","test"):
            a, c, direct_arr, truth_arr = load_direct(
                dataset,
                h,
                split,
            )

print("[PASS] Exp34R direct caches align exactly with all Exp33 manifests")


In [ ]:
# Variable-shadowing guard
# Dataset identifiers must remain strings before the expensive retrieval pass.

for dataset in DATASETS:
    assert isinstance(dataset, str)
    assert dataset in CHANNELS

print("[PASS] dataset identifiers are intact; no shadowing before retrieval.")


In [ ]:

# Cell 7 — per-channel retrieval inference with resumable channel parts

def memory_for_channel(z,ch,h,stack,boundary):
    memory_anchors=np.arange(
        cfg.retrieval_seq_len,
        int(boundary)-int(h)+1,
        cfg.memory_stride,
        dtype=np.int64,
    )

    if len(memory_anchors)<cfg.top_m_pattern:
        raise ValueError(
            f"insufficient memory ch={ch} H={h}: {len(memory_anchors)}"
        )

    pi=(
        memory_anchors[:,None]
        -cfg.retrieval_seq_len
        +np.arange(cfg.retrieval_seq_len)[None,:]
    )

    past=z[pi,ch].astype(np.float32)

    return {
        "anchors":memory_anchors,
        "pattern":batch_pattern(past),
        "stats":stack.scale_stats(context7(past)),
        "emb":encode_rows(stack.henc,past),
    }

@torch.no_grad()
def retrieve_channel(
    z,h,stack,boundary,
    query_anchor,ch,
    query_direct_abs,
    query_truth_abs,
):
    B=len(query_anchor)
    mem=memory_for_channel(z,ch,h,stack,boundary)

    pi=(
        query_anchor[:,None]
        -cfg.retrieval_seq_len
        +np.arange(cfg.retrieval_seq_len)[None,:]
    )
    past=z[pi,ch].astype(np.float32)

    qp=batch_pattern(past)
    qs=stack.scale_stats(context7(past))
    qe=encode_rows(stack.qenc,past)

    ps=np.einsum("bl,ml->bm",qp,mem["pattern"]).astype(np.float32)
    ls=np.einsum("be,me->bm",qe,mem["emb"]).astype(np.float32)

    retrieval_abs=np.empty((B,h),np.float32)
    analog_mse=np.empty(B,np.float32)
    union_size=np.empty(B,np.int16)
    learned_only=np.empty(B,np.int16)

    for left in range(0,B,cfg.query_batch):
        right=min(B,left+cfg.query_batch)
        bsz=right-left

        cand_idx=np.zeros((bsz,cfg.union_max),np.int64)
        valid=np.zeros((bsz,cfg.union_max),bool)

        psl=ps[left:right]
        lsl=ls[left:right]

        for j in range(bsz):
            ptop=topk_np(psl[j],cfg.top_m_pattern)
            ltop=topk_np(lsl[j],cfg.top_m_learned)

            union=np.union1d(ptop,ltop)

            union_size[left+j]=len(union)
            learned_only[left+j]=len(np.setdiff1d(ltop,ptop))

            if len(union)>cfg.union_max:
                op=np.argsort(-psl[j])
                ol=np.argsort(-lsl[j])
                rp=np.empty_like(op); rl=np.empty_like(ol)
                rp[op]=np.arange(1,len(op)+1)
                rl[ol]=np.arange(1,len(ol)+1)
                rrf=1./(60+rp)+1./(60+rl)
                union=union[np.argsort(-rrf[union])[:cfg.union_max]]

            n=len(union)
            cand_idx[j,:n]=union
            valid[j,:n]=True

        base_pattern=np.take_along_axis(
            psl,cand_idx,axis=1
        ).astype(np.float32)

        mstats=mem["stats"][cand_idx]
        memb=mem["emb"][cand_idx]

        qjoint=np.concatenate(
            [qs[left:right],qe[left:right]],
            axis=1,
        )
        cjoint=np.concatenate([mstats,memb],axis=2)
        qrep=np.repeat(qjoint[:,None,:],cfg.union_max,axis=1)

        feat=np.concatenate(
            [
                base_pattern[:,:,None],
                qrep,
                cjoint,
                qrep-cjoint,
                np.abs(qrep-cjoint),
            ],
            axis=2,
        ).astype(np.float32)

        score=stack.reranker(
            torch.from_numpy(feat).float().to(DEVICE),
            torch.from_numpy(base_pattern).float().to(DEVICE),
            torch.from_numpy(valid).to(DEVICE),
        ).cpu().numpy().astype(np.float32)

        top=np.argsort(-score,axis=1)[:,:cfg.top_k]
        selected_idx=np.take_along_axis(cand_idx,top,axis=1)
        selected_anchor=mem["anchors"][selected_idx]

        offsets=np.arange(h,dtype=np.int64)
        future=z[
            selected_anchor[:,:,None]+offsets[None,None,:],
            ch,
        ].astype(np.float32)

        current=z[selected_anchor-1,ch].astype(np.float32)
        cand_resid=future-current[:,:,None]

        qcur=z[query_anchor[left:right]-1,ch].astype(np.float32)
        retrieved_resid=cand_resid.mean(axis=1)
        retrieval_abs[left:right]=retrieved_resid+qcur[:,None]

        true_resid=(
            query_truth_abs[left:right]
            -qcur[:,None]
        )
        analog_mse[left:right]=(
            (cand_resid-true_resid[:,None,:])**2
        ).mean(axis=(1,2))

    return {
        "retrieval":retrieval_abs,
        "analog_mse":analog_mse,
        "union_size":union_size,
        "learned_only":learned_only,
    }

def part_path(d,h,split,ch):
    p=PART_ROOT/d/f"H{h}"/split
    p.mkdir(parents=True,exist_ok=True)
    return p/f"ch{ch:04d}.npz"

def part_valid(path,anchors,h):
    if not path.exists():
        return False
    try:
        with np.load(path,allow_pickle=False) as z:
            return (
                np.array_equal(z["anchor"],anchors)
                and z["retrieval"].shape==(len(anchors),h)
                and z["analog_mse"].shape==(len(anchors),)
            )
    except Exception:
        return False

print("[PASS] retrieval inference functions ready")


In [ ]:

# Cell 8 — run 24 × val/test PRR retrieval, resumably

summary_rows=[]

for d in DATASETS:
    z=DATA[d]["z"]

    for h in HORIZONS:
        print("\n"+"="*110)
        print(d,"H=",h)
        print("="*110)

        stack=load_stack(STACKS[(d,h)], expected_dataset=d, expected_horizon=h)
        assert stack.dataset.lower()==d.lower() and stack.horizon==h

        split_outputs={}

        for split in ("val","test"):
            a,c,direct,truth=load_direct(d,h,split)

            boundary=(
                DATA[d]["train_end"]
                if split=="val"
                else DATA[d]["val_end"]
            )

            retrieval=np.empty_like(direct)
            analog=np.empty(len(a),np.float32)
            union_sz=np.empty(len(a),np.int16)
            learned_only=np.empty(len(a),np.int16)

            used_channels=np.unique(c)

            t0=time.perf_counter()

            for ci,ch in enumerate(used_channels):
                rows=np.flatnonzero(c==ch)
                aa=a[rows]
                pp=part_path(d,h,split,int(ch))

                if part_valid(pp,aa,h):
                    with np.load(pp,allow_pickle=False) as zz:
                        retrieval[rows]=zz["retrieval"]
                        analog[rows]=zz["analog_mse"]
                        union_sz[rows]=zz["union_size"]
                        learned_only[rows]=zz["learned_only"]
                else:
                    out=retrieve_channel(
                        z,h,stack,boundary,
                        aa,int(ch),
                        direct[rows],
                        truth[rows],
                    )

                    retrieval[rows]=out["retrieval"]
                    analog[rows]=out["analog_mse"]
                    union_sz[rows]=out["union_size"]
                    learned_only[rows]=out["learned_only"]

                    tmp=pp.with_suffix(".tmp.npz")
                    np.savez_compressed(
                        tmp,
                        anchor=aa,
                        retrieval=out["retrieval"],
                        analog_mse=out["analog_mse"],
                        union_size=out["union_size"],
                        learned_only=out["learned_only"],
                    )
                    os.replace(tmp,pp)

                if (
                    ci==0
                    or (ci+1)%100==0
                    or ci+1==len(used_channels)
                ):
                    print(
                        f"  {split}: channel {ci+1}/{len(used_channels)}"
                    )

            sec=time.perf_counter()-t0

            split_outputs[split]={
                "anchor":a,
                "channel":c,
                "direct":direct,
                "truth":truth,
                "retrieval":retrieval,
                "analog":analog,
                "union_size":union_sz,
                "learned_only":learned_only,
            }

            print(
                f"  {split} complete: pairs={len(a)}, "
                f"AnalogFutureMSE={analog.mean():.6f}, "
                f"seconds={sec:.1f}"
            )

        # ----- validation-selected global analytic beta -----
        v=split_outputs["val"]
        e=(v["direct"]-v["truth"]).astype(np.float64)
        delta=(v["retrieval"]-v["direct"]).astype(np.float64)

        denom=float(np.sum(delta*delta))
        numer=float(np.sum(e*delta))

        beta_global=(
            float(np.clip(-numer/(denom+1e-12),0.,1.))
            if denom>0 else 0.0
        )

        # ----- final test metrics -----
        t=split_outputs["test"]

        fixed01=(
            t["direct"]
            +0.1*(t["retrieval"]-t["direct"])
        )
        global_pred=(
            t["direct"]
            +beta_global*(t["retrieval"]-t["direct"])
        )

        def mse(x,y):
            return float(np.mean(
                (x.astype(np.float64)-y.astype(np.float64))**2
            ))

        def mae(x,y):
            return float(np.mean(
                np.abs(x.astype(np.float64)-y.astype(np.float64))
            ))

        direct_mse=mse(t["direct"],t["truth"])
        retrieval_mse=mse(t["retrieval"],t["truth"])
        fixed_mse=mse(fixed01,t["truth"])
        global_mse=mse(global_pred,t["truth"])

        row={
            "dataset":d,
            "horizon":h,
            "val_analog_future_mse":float(v["analog"].mean()),
            "test_analog_future_mse":float(t["analog"].mean()),
            "mean_union_size":float(t["union_size"].mean()),
            "mean_learned_only_candidates":float(t["learned_only"].mean()),
            "beta_global":beta_global,
            "direct_mse":direct_mse,
            "retrieval_only_mse":retrieval_mse,
            "fixed01_mse":fixed_mse,
            "global_mse":global_mse,
            "direct_mae":mae(t["direct"],t["truth"]),
            "global_mae":mae(global_pred,t["truth"]),
            "fixed01_gain_pct":100*(direct_mse-fixed_mse)/direct_mse,
            "global_gain_pct":100*(direct_mse-global_mse)/direct_mse,
        }
        summary_rows.append(row)

        print(
            f"  TEST Direct={direct_mse:.6f} "
            f"Fixed0.1={fixed_mse:.6f} "
            f"Global={global_mse:.6f} "
            f"beta={beta_global:.5f} "
            f"gain={row['global_gain_pct']:+.3f}%"
        )

        # persist each condition immediately
        pd.DataFrame(summary_rows).to_csv(
            RESULT_ROOT/"patchtst_prr_running_results.csv",
            index=False,
        )

        del stack
        gc.collect()
        torch.cuda.empty_cache()

result_df=pd.DataFrame(summary_rows).sort_values(
    ["dataset","horizon"]
).reset_index(drop=True)

display(result_df)

assert len(result_df)==24
result_df.to_csv(
    RESULT_ROOT/"patchtst_24_prr_results.csv",
    index=False,
)

print("[PASS] 24/24 PatchTST PRR retrieval conditions complete")


In [ ]:

# Cell 9 — compact result summary

df=result_df.copy()

df["win_global"]=df.global_mse < df.direct_mse - 1e-12
df["tie_global"]=np.isclose(
    df.global_mse,
    df.direct_mse,
    rtol=0,
    atol=1e-12,
)
df["loss_global"]=df.global_mse > df.direct_mse + 1e-12

dataset_summary=(
    df.groupby("dataset",sort=False)
    .agg(
        conditions=("horizon","size"),
        wins=("win_global","sum"),
        mean_gain_pct=("global_gain_pct","mean"),
        mean_beta=("beta_global","mean"),
        retrieval_mse=("test_analog_future_mse","mean"),
    )
    .reset_index()
)

display(dataset_summary)

wins=int(df.win_global.sum())
ties=int(df.tie_global.sum())
losses=int(df.loss_global.sum())

print("="*100)
print("PRR + validation GlobalAnalytic vs Direct")
print(f"Wins / Ties / Losses: {wins} / {ties} / {losses}")
print(f"Mean MSE gain: {df.global_gain_pct.mean():+.3f}%")
print(f"Median MSE gain: {df.global_gain_pct.median():+.3f}%")
print("="*100)

print("\nLargest gains:")
display(
    df.sort_values("global_gain_pct",ascending=False)[
        ["dataset","horizon","direct_mse","global_mse",
         "beta_global","global_gain_pct"]
    ].head(10)
)

print("\nLargest degradations:")
display(
    df.sort_values("global_gain_pct",ascending=True)[
        ["dataset","horizon","direct_mse","global_mse",
         "beta_global","global_gain_pct"]
    ].head(10)
)

dataset_summary.to_csv(
    RESULT_ROOT/"patchtst_prr_dataset_summary.csv",
    index=False,
)


In [ ]:

# Cell 10 — handoff for PRR-Stat and final 120-condition expansion

# Cell 9 creates df = result_df.copy() and adds the win/tie/loss flags there.
# Use df consistently in the final summary.
assert "df" in globals(), "Run the previous result-summary cell first."
required_cols = {
    "win_global","tie_global","loss_global",
    "global_gain_pct","beta_global"
}
missing_cols = required_cols - set(df.columns)
assert not missing_cols, f"Missing summary columns: {sorted(missing_cols)}"

summary={
    "experiment":"Exp35BR3",
    "backbone":"PatchTST",
    "conditions":int(len(df)),
    "wins_vs_direct":int(df["win_global"].sum()),
    "ties_vs_direct":int(df["tie_global"].sum()),
    "losses_vs_direct":int(df["loss_global"].sum()),
    "mean_gain_pct":float(df["global_gain_pct"].mean()),
    "median_gain_pct":float(df["global_gain_pct"].median()),
    "mean_validation_beta":float(df["beta_global"].mean()),
    "forecaster_training_performed":False,
    "forecaster_forward_performed":False,
    "direct_source":"Exp34R cache",
    "query_protocol":"Exp33",
    "retrieval_stack_coverage":"24/24",
}

(RESULT_ROOT/"exp35B_summary.json").write_text(
    json.dumps(summary,indent=2),
    encoding="utf-8",
)

print(json.dumps(summary,indent=2))

print("\n"+"="*100)
print("EXP35BR3 COMPLETE")
print("="*100)
print("We now have the first 24/120 final-protocol forecasting results.")
print("NEXT:")
print("1. Re-evaluate PRR-Stat on the same Exp33 pairs.")
print("2. Compare PRR-Stat vs PRR retrieval and forecasting.")
print("3. Reuse the same retrieval outputs with the remaining four forecaster caches.")
print("4. Proceed to frozen-forecaster integration and validation-only trust calibration.")
